Proyecto Final

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Instalar Java 8 y pyspark
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!pip install pyspark

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql import Window

# Crear la sesión de Spark
spark = SparkSession.builder \
    .appName("Pract2_PySpark") \
    .getOrCreate()

# Verificar la versión
print("Versión de Spark:", spark.version)

sc = spark.sparkContext

DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/FinPlus/"

E: Failed to fetch http://security.ubuntu.com/ubuntu/pool/universe/o/openjdk-8/openjdk-8-jre-headless_8u462-ga%7eus1-0ubuntu2%7e22.04.2_amd64.deb  404  Not Found [IP: 185.125.190.81 80]
E: Failed to fetch http://security.ubuntu.com/ubuntu/pool/universe/o/openjdk-8/openjdk-8-jdk-headless_8u462-ga%7eus1-0ubuntu2%7e22.04.2_amd64.deb  404  Not Found [IP: 185.125.190.81 80]
E: Unable to fetch some archives, maybe run apt-get update or try with --fix-missing?
Versión de Spark: 3.5.1


# Impotamos la data

In [ ]:
client = (spark.read.option('header', 'true').option('delimiter',',')
                     .csv(DATA_PATH + 'CLIENTS.csv'))
client.show(5, 0)

+------------+----------------------+-----------------+------+------------+--------------+-----------+--------------+--------------+--------------------+------------------+------------------+-------------+--------------+-----------+-----------------+-------+-----------+------------------+------------------+------------------+---------------------+------------------+----------+--------------+----------+--------------------------+--------+---------------------+------------------------+------------------------+------------------------+---------------------------+---------------------------+---------------------------+-----------------------+-----------------------+-----------------------+----------------------+----------------------+-------------------+---------------------+-----------------+-------------------+----------------+
|CLIENT_ID   |NON_COMPLIANT_CONTRACT|NAME_PRODUCT_TYPE|GENDER|TOTAL_INCOME|AMOUNT_PRODUCT|INSTALLMENT|EDUCATION     |MARITAL_STATUS|HOME_SITUATION      |REGION_SC

In [ ]:
beh = (spark.read.parquet(DATA_PATH + 'BEHAVIOURAL_PARQUET'))
beh.show(5, 0)

+------------------+------------+----------+--------------------+-----------------+------------------------+--------------------+------------------------+--------------------------+-------------------+-------------------+---------------+------------------+--------+
|CONTRACT_ID       |CLIENT_ID   |DATE      |CREDICT_CARD_BALANCE|CREDIT_CARD_LIMIT|CREDIT_CARD_DRAWINGS_ATM|CREDIT_CARD_DRAWINGS|CREDIT_CARD_DRAWINGS_POS|CREDIT_CARD_DRAWINGS_OTHER|CREDIT_CARD_PAYMENT|NUMBER_DRAWINGS_ATM|NUMBER_DRAWINGS|NUMBER_INSTALMENTS|CURRENCY|
+------------------+------------+----------+--------------------+-----------------+------------------------+--------------------+------------------------+--------------------------+-------------------+-------------------+---------------+------------------+--------+
|ES1821000018d00XXX|ES182394447V|2021-08-29|491.21              |540.0            |0.0                     |28.03               |28.03                   |0.0                       |46.81              |0

# Quitamos los duplicados en los dos data sets

In [ ]:
client_sin_duplicados_por_id = client.dropDuplicates(['CLIENT_ID'])

client_sin_duplicados_por_id.show(5)

print(f"Filas originales en client: {client.count()}")
print(f"Filas después de eliminar duplicados (por CLIENT_ID) en client: {client_sin_duplicados_por_id.count()}")

+------------+----------------------+-----------------+------+------------+--------------+-----------+---------+--------------+--------------+------------+-----------------+-------------+--------------+-----------+-----------------+-------+-----------+------------------+------------------+------------------+---------------------+------------------+----------+--------------+----------+--------------------------+--------+---------------------+------------------------+------------------------+------------------------+---------------------------+---------------------------+---------------------------+-----------------------+-----------------------+-----------------------+----------------------+----------------------+-------------------+---------------------+-----------------+-------------------+----------------+
|   CLIENT_ID|NON_COMPLIANT_CONTRACT|NAME_PRODUCT_TYPE|GENDER|TOTAL_INCOME|AMOUNT_PRODUCT|INSTALLMENT|EDUCATION|MARITAL_STATUS|HOME_SITUATION|REGION_SCORE|     AGE_IN_YEARS|JOB_SEN

In [ ]:
beh_sin_duplicados_por_id = beh.dropDuplicates(['CLIENT_ID', 'DATE', 'CREDICT_CARD_BALANCE', 'CONTRACT_ID'])

beh_sin_duplicados_por_id.show(5)

print(f"Filas originales en beh: {beh.count()}")
print(f"Filas después de eliminar duplicados (por CLIENT_ID) en beh: {beh_sin_duplicados_por_id.count()}")

+------------------+------------+----------+--------------------+-----------------+------------------------+--------------------+------------------------+--------------------------+-------------------+-------------------+---------------+------------------+--------+
|       CONTRACT_ID|   CLIENT_ID|      DATE|CREDICT_CARD_BALANCE|CREDIT_CARD_LIMIT|CREDIT_CARD_DRAWINGS_ATM|CREDIT_CARD_DRAWINGS|CREDIT_CARD_DRAWINGS_POS|CREDIT_CARD_DRAWINGS_OTHER|CREDIT_CARD_PAYMENT|NUMBER_DRAWINGS_ATM|NUMBER_DRAWINGS|NUMBER_INSTALMENTS|CURRENCY|
+------------------+------------+----------+--------------------+-----------------+------------------------+--------------------+------------------------+--------------------------+-------------------+-------------------+---------------+------------------+--------+
|ES1821489396v00XXX|ES182100006A|2021-08-29|                 0.0|           3240.0|                     0.0|                 0.0|                     0.0|                       0.0|                0.0| 

# Contamos el número de nulls que hay en cada una de las columnas

In [ ]:
for column in client.columns:
    null_count = client.filter(client[column].isNull()).count()
    print(f"Columna '{column}': {null_count} valores nulos")

Columna 'CLIENT_ID': 0 valores nulos
Columna 'NON_COMPLIANT_CONTRACT': 0 valores nulos
Columna 'NAME_PRODUCT_TYPE': 0 valores nulos
Columna 'GENDER': 0 valores nulos
Columna 'TOTAL_INCOME': 0 valores nulos
Columna 'AMOUNT_PRODUCT': 0 valores nulos
Columna 'INSTALLMENT': 7 valores nulos
Columna 'EDUCATION': 39640 valores nulos
Columna 'MARITAL_STATUS': 2 valores nulos
Columna 'HOME_SITUATION': 0 valores nulos
Columna 'REGION_SCORE': 0 valores nulos
Columna 'AGE_IN_YEARS': 0 valores nulos
Columna 'JOB_SENIORITY': 29174 valores nulos
Columna 'HOME_SENIORITY': 0 valores nulos
Columna 'LAST_UPDATE': 0 valores nulos
Columna 'OWN_INSURANCE_CAR': 0 valores nulos
Columna 'CAR_AGE': 107550 valores nulos
Columna 'FAMILY_SIZE': 2 valores nulos
Columna 'REACTIVE_SCORING': 91901 valores nulos
Columna 'PROACTIVE_SCORING': 337 valores nulos
Columna 'BEHAVIORAL_SCORING': 32246 valores nulos
Columna 'DAYS_LAST_INFO_CHANGE': 1 valores nulos
Columna 'NUMBER_OF_PRODUCTS': 21903 valores nulos
Columna 'OCCUP

In [ ]:
for column in beh.columns:
    null_count = beh.filter(beh[column].isNull()).count()
    print(f"Columna '{column}': {null_count} valores nulos")

In [ ]:
client.printSchema()

root
 |-- CLIENT_ID: string (nullable = true)
 |-- NON_COMPLIANT_CONTRACT: string (nullable = true)
 |-- NAME_PRODUCT_TYPE: string (nullable = true)
 |-- GENDER: string (nullable = true)
 |-- TOTAL_INCOME: string (nullable = true)
 |-- AMOUNT_PRODUCT: string (nullable = true)
 |-- INSTALLMENT: string (nullable = true)
 |-- EDUCATION: string (nullable = true)
 |-- MARITAL_STATUS: string (nullable = true)
 |-- HOME_SITUATION: string (nullable = true)
 |-- REGION_SCORE: string (nullable = true)
 |-- AGE_IN_YEARS: string (nullable = true)
 |-- JOB_SENIORITY: string (nullable = true)
 |-- HOME_SENIORITY: string (nullable = true)
 |-- LAST_UPDATE: string (nullable = true)
 |-- OWN_INSURANCE_CAR: string (nullable = true)
 |-- CAR_AGE: string (nullable = true)
 |-- FAMILY_SIZE: string (nullable = true)
 |-- REACTIVE_SCORING: string (nullable = true)
 |-- PROACTIVE_SCORING: string (nullable = true)
 |-- BEHAVIORAL_SCORING: string (nullable = true)
 |-- DAYS_LAST_INFO_CHANGE: string (nullable = 

In [ ]:
beh.printSchema()

In [ ]:
client.columns

['CLIENT_ID',
 'NON_COMPLIANT_CONTRACT',
 'NAME_PRODUCT_TYPE',
 'GENDER',
 'TOTAL_INCOME',
 'AMOUNT_PRODUCT',
 'INSTALLMENT',
 'EDUCATION',
 'MARITAL_STATUS',
 'HOME_SITUATION',
 'REGION_SCORE',
 'AGE_IN_YEARS',
 'JOB_SENIORITY',
 'HOME_SENIORITY',
 'LAST_UPDATE',
 'OWN_INSURANCE_CAR',
 'CAR_AGE',
 'FAMILY_SIZE',
 'REACTIVE_SCORING',
 'PROACTIVE_SCORING',
 'BEHAVIORAL_SCORING',
 'DAYS_LAST_INFO_CHANGE',
 'NUMBER_OF_PRODUCTS',
 'OCCUPATION',
 'DIGITAL_CLIENT',
 'HOME_OWNER',
 'EMPLOYER_ORGANIZATION_TYPE',
 'CURRENCY',
 'NUM_PREVIOUS_LOAN_APP',
 'LOAN_ANNUITY_PAYMENT_MAX',
 'LOAN_ANNUITY_PAYMENT_MIN',
 'LOAN_ANNUITY_PAYMENT_SUM',
 'LOAN_APPLICATION_AMOUNT_MAX',
 'LOAN_APPLICATION_AMOUNT_MIN',
 'LOAN_APPLICATION_AMOUNT_SUM',
 'LOAN_CREDIT_GRANTED_MAX',
 'LOAN_CREDIT_GRANTED_MIN',
 'LOAN_CREDIT_GRANTED_SUM',
 'LOAN_VARIABLE_RATE_MAX',
 'LOAN_VARIABLE_RATE_MIN',
 'NUM_STATUS_ANNULLED',
 'NUM_STATUS_AUTHORIZED',
 'NUM_STATUS_DENIED',
 'NUM_STATUS_NOT_USED',
 'NUM_FLAG_INSURED']

In [ ]:
beh.columns

['CONTRACT_ID',
 'CLIENT_ID',
 'DATE',
 'CREDICT_CARD_BALANCE',
 'CREDIT_CARD_LIMIT',
 'CREDIT_CARD_DRAWINGS_ATM',
 'CREDIT_CARD_DRAWINGS',
 'CREDIT_CARD_DRAWINGS_POS',
 'CREDIT_CARD_DRAWINGS_OTHER',
 'CREDIT_CARD_PAYMENT',
 'NUMBER_DRAWINGS_ATM',
 'NUMBER_DRAWINGS',
 'NUMBER_INSTALMENTS',
 'CURRENCY']

# Cambiamos Nulls

In [ ]:
df_client = client

score_columns = ['REACTIVE_SCORING', 'PROACTIVE_SCORING', 'BEHAVIORAL_SCORING']

for col_name in score_columns:
    df_client = df_client.withColumn(col_name, F.col(col_name).cast(T.FloatType()))

avg_reactive_scoring = df_client.filter(F.col('REACTIVE_SCORING').isNotNull()).agg(F.avg('REACTIVE_SCORING')).collect()[0][0]
avg_proactive_scoring = df_client.filter(F.col('PROACTIVE_SCORING').isNotNull()).agg(F.avg('PROACTIVE_SCORING')).collect()[0][0]
avg_behavioral_scoring = df_client.filter(F.col('BEHAVIORAL_SCORING').isNotNull()).agg(F.avg('BEHAVIORAL_SCORING')).collect()[0][0]

fill_values = {
    'INSTALLMENT': 0.0,
    'EDUCATION': 'Bachelor',
    'MARITAL_STATUS': 'NA',
    'JOB_SENIORITY': 0.0,
    'CAR_AGE': 0.0,
    'FAMILY_SIZE': 1.0,
    'REACTIVE_SCORING': avg_reactive_scoring,
    'PROACTIVE_SCORING': avg_proactive_scoring,
    'BEHAVIORAL_SCORING': avg_behavioral_scoring,
    'DAYS_LAST_INFO_CHANGE': 0.0,
    'NUMBER_OF_PRODUCTS': 0.0,
    'EMPLOYER_ORGANIZATION_TYPE': 'NA',
    'NUM_PREVIOUS_LOAN_APP': 0.0,
    'LOAN_ANNUITY_PAYMENT_MAX': 0.0,
    'LOAN_ANNUITY_PAYMENT_MIN': 0.0,
    'LOAN_ANNUITY_PAYMENT_SUM': 0.0,
    'LOAN_APPLICATION_AMOUNT_MAX': 0.0,
    'LOAN_APPLICATION_AMOUNT_MIN': 0.0,
    'LOAN_APPLICATION_AMOUNT_SUM': 0.0,
    'LOAN_CREDIT_GRANTED_MAX': 0.0,
    'LOAN_CREDIT_GRANTED_MIN': 0.0,
    'LOAN_CREDIT_GRANTED_SUM': 0.0,
    'LOAN_VARIABLE_RATE_MAX': 0.0,
    'LOAN_VARIABLE_RATE_MIN': 0.0,
    'NUM_STATUS_ANNULLED': 0.0,
    'NUM_STATUS_AUTHORIZED': 0.0,
    'NUM_STATUS_DENIED': 0.0,
    'NUM_STATUS_NOT_USED': 0.0,
    'NUM_FLAG_INSURED': 0.0
}

df_client = df_client.na.fill(fill_values)

print("Client DataFrame after handling missing values. Verifying null counts:")
for column in df_client.columns:
    null_count = df_client.filter(F.col(column).isNull()).count()
    if null_count > 0:
        print(f"Columna '{column}': {null_count} valores nulos")
    else:
        print(f"Columna '{column}': 0 valores nulos")

df_client.show(5)

Client DataFrame after handling missing values. Verifying null counts:
Columna 'CLIENT_ID': 0 valores nulos
Columna 'NON_COMPLIANT_CONTRACT': 0 valores nulos
Columna 'NAME_PRODUCT_TYPE': 0 valores nulos
Columna 'GENDER': 0 valores nulos
Columna 'TOTAL_INCOME': 0 valores nulos
Columna 'AMOUNT_PRODUCT': 0 valores nulos
Columna 'INSTALLMENT': 0 valores nulos
Columna 'EDUCATION': 0 valores nulos
Columna 'MARITAL_STATUS': 0 valores nulos
Columna 'HOME_SITUATION': 0 valores nulos
Columna 'REGION_SCORE': 0 valores nulos
Columna 'AGE_IN_YEARS': 0 valores nulos
Columna 'JOB_SENIORITY': 0 valores nulos
Columna 'HOME_SENIORITY': 0 valores nulos
Columna 'LAST_UPDATE': 0 valores nulos
Columna 'OWN_INSURANCE_CAR': 0 valores nulos
Columna 'CAR_AGE': 0 valores nulos
Columna 'FAMILY_SIZE': 0 valores nulos
Columna 'REACTIVE_SCORING': 0 valores nulos
Columna 'PROACTIVE_SCORING': 0 valores nulos
Columna 'BEHAVIORAL_SCORING': 0 valores nulos
Columna 'DAYS_LAST_INFO_CHANGE': 0 valores nulos
Columna 'NUMBER_

# Transformacion de la data

In [ ]:
(df_client.groupBy('NAME_PRODUCT_TYPE')
    .count()
    .orderBy(F.desc('count'))
).show(10)

+-----------------+------+
|NAME_PRODUCT_TYPE| count|
+-----------------+------+
|        PRODUCT 1|147470|
|        PRODUCT 2| 15507|
+-----------------+------+



In [ ]:
(client.groupBy(
    'DIGITAL_CLIENT').count().orderBy(F.desc('count'))
).show(10)

+--------------+------+
|DIGITAL_CLIENT| count|
+--------------+------+
|             0|153832|
|             1|  9145|
+--------------+------+



In [ ]:
(client.filter(client['DIGITAL_CLIENT'] == '1').count())/(client.count())*100


5.6112212152635035

In [ ]:
df = client.join(beh, "CLIENT_ID")

# Convert necessary columns to numeric types and handle nulls with 0
df = df.withColumn('INSTALLMENT', F.col('INSTALLMENT').cast('float')) \
       .withColumn('CREDICT_CARD_BALANCE', F.col('CREDICT_CARD_BALANCE').cast('float')) \
       .withColumn('TOTAL_INCOME', F.col('TOTAL_INCOME').cast('float'))

# Fill any remaining nulls in these columns with 0 after casting, before calculation
df = df.na.fill({'INSTALLMENT': 0.0, 'CREDICT_CARD_BALANCE': 0.0, 'TOTAL_INCOME': 0.0})

df = df.select(
    F.col('INSTALLMENT'),
    F.col('CREDICT_CARD_BALANCE'),
    F.col('TOTAL_INCOME'),
    (((F.col('INSTALLMENT') + (F.col('CREDICT_CARD_BALANCE') * 0.05)) / F.col('TOTAL_INCOME')) * 100).alias('DEBT_RATIO')
).withColumn(
    "Riesgo",
    F.when(F.col("DEBT_RATIO") > 45, "Alto Riesgo")
    .otherwise("Bajo Riesgo")
)

df.show()

(df.groupBy(
    'Riesgo').count().orderBy(F.desc('count'))
).show(10)


+-----------+--------------------+------------+------------------+-----------+
|INSTALLMENT|CREDICT_CARD_BALANCE|TOTAL_INCOME|        DEBT_RATIO|     Riesgo|
+-----------+--------------------+------------+------------------+-----------+
|     219.19|              491.21|      1188.0|20.517719024760954|Bajo Riesgo|
|     219.19|              466.55|      1188.0|20.413931130560158|Bajo Riesgo|
|     219.19|             1640.35|      1188.0|25.354166769419457|Bajo Riesgo|
|     219.19|             1366.35|      1188.0|24.200968116220803|Bajo Riesgo|
|     219.19|              532.33|      1188.0|20.690783105715358|Bajo Riesgo|
|     393.34|             1203.17|      1620.0| 27.99373447747878|Bajo Riesgo|
|     393.34|               187.0|      1620.0|24.857407181351274|Bajo Riesgo|
|     393.34|             1236.33|      1620.0| 28.09607988522377|Bajo Riesgo|
|     393.34|             1190.34|      1620.0| 27.95413547092014|Bajo Riesgo|
|     393.34|              710.26|      1620.0| 26.4

In [ ]:
beh = (beh.withColumn('CREDIT_CARD_DRAWINGS_ATM', beh['CREDIT_CARD_DRAWINGS_ATM'].cast('float'))\
      .withColumn('CREDIT_CARD_DRAWINGS_POS', beh['CREDIT_CARD_DRAWINGS_POS'].cast('float'))\
      .withColumn('CREDIT_CARD_DRAWINGS', beh['CREDIT_CARD_DRAWINGS'].cast('float')))

# Agrupar y sumar
df_totals = beh.groupBy("CLIENT_ID").agg(
    F.sum("CREDIT_CARD_DRAWINGS_ATM").alias("total_atm"),
    F.sum("CREDIT_CARD_DRAWINGS_POS").alias("total_pos"),
    F.sum("CREDIT_CARD_DRAWINGS").alias("total_gastos")
)

# Calcular porcentajes
df_percentages = df_totals.withColumn(
    "pct_atm",
    (F.col("total_atm") / F.col("total_gastos")) * 100
).withColumn(
    "pct_pos",
    (F.col("total_pos") / F.col("total_gastos")) * 100
).fillna(0)

# Segmentar
df_segmented = df_percentages.withColumn(
    "segmento",
    F.when(F.col("pct_atm") > 70, "Efectivo-dependiente")
    .when(F.col("pct_pos") > 70, "Cashless")
    .otherwise("Mixto")
)

# Mostrar resultados
df_segmented.show()

+------------+------------------+------------------+------------------+------------------+------------------+--------------------+
|   CLIENT_ID|         total_atm|         total_pos|      total_gastos|           pct_atm|           pct_pos|            segmento|
+------------+------------------+------------------+------------------+------------------+------------------+--------------------+
|ES182405039N|10341.000030517578|109.79999828338623|10450.800025939941| 98.94936277462176|1.0506372527543493|Efectivo-dependiente|
|ES182245476M|               0.0|               0.0|               0.0|               0.0|               0.0|               Mixto|
|ES182319379G| 1792.800048828125|1217.5900192260742|3010.3900451660156|59.553746256467456|40.446254503838794|               Mixto|
|ES182391573W|               0.0|               0.0|               0.0|               0.0|               0.0|               Mixto|
|ES182189508S|            3105.0|               0.0|            3105.0|            